In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
#store history
chat_history=[]

In [ ]:

# 1. Initialize history with a System Prompt to define behavior
messages = [
    {"role": "system", "content": "You are a helpful and concise chatbot."},
]

print("Chatbot: Hello! Type 'exit' to stop.")

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye!")
        break

    # 2. Add user input to history
    messages.append({"role": "user", "content": user_input})

    # 3. Apply the specific TinyLlama chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 4. Generate response
    output = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2 # Prevents the model from repeating itself
    )

    # Decode only the new tokens (the assistant's response)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"Chatbot: {response}")

    # 5. Add assistant response to history so the model "remembers"
    messages.append({"role": "assistant", "content": response})

Chatbot: Hello! Type 'exit' to stop.
User: how many continents are there in the world?
Chatbot: There are no actual continents, as such. The term "continent" refers to geographic landmasses that have historically been inhabited by humans for thousands of years or longer (such as Europe, Asia, Africa, North America, South America, Australia, Antarctica). In modern times, however, most maps use contours and lines of latitude to represent these large land masses into smaller regions within those larger landmasses. Therefore, while there may not be specific, formal boundaries separating different parts of the Earth's surface, they do exist on paper and can vary based on differing regional perspectives and definitions.
User: how to make a pizza
Chatbot: Sure! Here is a simple recipe for making homemade pizzas:

Ingredients:
- 1 sheet of premade pizza dough (think frozen dough from your grocery store)
- Fresh mozzarella cheese
- Chopped fresh basil leaves
- Sliced tomatoes (any color works!)